[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/01_tag_1_machine_learning_playground.ipynb)

# Tag 1 Machine Learning Playground

Dieses Notebook begleitet Tag 1 praktisch. Es zeigt Regression, Klassifikation, Clustering, Metriken, Train/Valid/Test-Splits und den Bias-Varianz-Tradeoff mit kleinen Datensätzen ohne Internet-Downloads.

## 1. Setup und Grundbegriffe

- **Sample**: eine Beobachtung, also eine Zeile in der Datentabelle.
- **Feature**: eine Eingabevariable, also eine Spalte in `X`.
- **Label/Zielgröße**: die gesuchte Ausgabe `y`.
- **Modellparameter**: Werte, die das Modell beim Training lernt.
- **Hyperparameter**: Einstellungen, die wir vor dem Training wählen, z. B. Baumtiefe oder Anzahl Nachbarn.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import ipywidgets as widgets

from sklearn.datasets import make_regression, make_classification, make_blobs, load_diabetes, load_breast_cancer, load_digits, load_wine
from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score, median_absolute_error,
    mean_absolute_percentage_error, accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score, roc_auc_score, average_precision_score,
    matthews_corrcoef, confusion_matrix, ConfusionMatrixDisplay, classification_report,
    RocCurveDisplay, PrecisionRecallDisplay, silhouette_score, adjusted_rand_score,
    normalized_mutual_info_score
)
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.style.use('default')
plt.rcParams['figure.figsize'] = (7, 4)
print('Setup abgeschlossen.')

In [ ]:
# Kleiner synthetischer Datensatz: X enthält Features, y die Zielgröße.
X_demo, y_demo = make_regression(n_samples=80, n_features=2, noise=12, random_state=RANDOM_STATE)
demo_df = pd.DataFrame(X_demo, columns=['Feature 1', 'Feature 2'])
demo_df['Zielgröße y'] = y_demo

display(demo_df.head())
print('X-Form:', X_demo.shape, '| y-Form:', y_demo.shape)

plt.scatter(demo_df['Feature 1'], demo_df['Zielgröße y'], c=demo_df['Feature 2'], cmap='viridis')
plt.colorbar(label='Feature 2')
plt.xlabel('Feature 1')
plt.ylabel('Zielgröße y')
plt.title('Samples, Features und Zielgröße')
plt.show()

### Live-Demo: Rohdaten selbst verändern

Bevor ein Modell trainiert wird, lohnt sich ein Blick auf die Rohdaten. In dieser Box kannst du Anzahl der Samples, Rauschen und Zusammenhang der Features verändern. Beobachte, wann eine klare Struktur sichtbar ist und wann die Daten schwer zu modellieren werden.

In [ ]:
def rohdaten_demo(samples=100, rauschen=12, feature_staerke=40):
    X_raw, y_raw = make_regression(
        n_samples=samples,
        n_features=2,
        n_informative=2,
        noise=rauschen,
        random_state=RANDOM_STATE,
    )
    y_raw = y_raw + feature_staerke * np.sin(X_raw[:, 0])
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].scatter(X_raw[:, 0], y_raw, c=X_raw[:, 1], cmap='viridis', alpha=0.75)
    axes[0].set_title('Feature 1 gegen Zielgröße')
    axes[0].set_xlabel('Feature 1'); axes[0].set_ylabel('y')
    axes[1].scatter(X_raw[:, 0], X_raw[:, 1], c=y_raw, cmap='coolwarm', alpha=0.75)
    axes[1].set_title('Rohdaten im Feature-Raum')
    axes[1].set_xlabel('Feature 1'); axes[1].set_ylabel('Feature 2')
    plt.tight_layout(); plt.show()

widgets.interact(
    rohdaten_demo,
    samples=widgets.IntSlider(value=100, min=30, max=300, step=10),
    rauschen=widgets.IntSlider(value=12, min=0, max=60, step=2),
    feature_staerke=widgets.IntSlider(value=40, min=0, max=120, step=10),
);

## 2. Überwachtes Lernen als Regression

Regression sagt numerische Werte voraus. Wir nutzen `load_diabetes` und zusätzlich einen kleinen synthetischen Mietpreis-Datensatz.

### Datensätze in diesem Kapitel

- **Diabetes (`load_diabetes`)**: kleiner scikit-learn-Datensatz mit medizinischen Messwerten. Die Zielgröße ist ein numerischer Krankheitsprogressionswert. Er eignet sich gut, um Regressionsmetriken zu vergleichen.
- **Synthetische Mietpreise**: bewusst einfacher Unterrichtsdatensatz mit Wohnfläche, Zimmern und Entfernung. Hier können wir Rauschen, Ausreißer und Modellkomplexität kontrolliert verändern.

In [ ]:
diabetes = load_diabetes(as_frame=True)
X = diabetes.data
y = diabetes.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=RANDOM_STATE)

models_reg = {
    'LinearRegression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0, random_state=RANDOM_STATE),
    'DecisionTreeRegressor': DecisionTreeRegressor(max_depth=4, random_state=RANDOM_STATE),
    'RandomForestRegressor': RandomForestRegressor(n_estimators=80, max_depth=5, random_state=RANDOM_STATE, n_jobs=-1),
}

def regression_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {
        'MAE': mean_absolute_error(y_true, y_pred),
        'MSE': mse,
        'RMSE': np.sqrt(mse),
        'R2': r2_score(y_true, y_pred),
        'MedianAE': median_absolute_error(y_true, y_pred),
        'MAPE': mean_absolute_percentage_error(y_true, y_pred),
    }

rows = []
for name, model in models_reg.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    cv = cross_val_score(model, X, y, cv=5, scoring='neg_mean_absolute_error')
    row = {'Modell': name, 'CV_MAE': -cv.mean()}
    row.update(regression_metrics(y_test, pred))
    rows.append(row)
reg_results = pd.DataFrame(rows).sort_values('RMSE')
display(reg_results)

**Metriken kurz erklärt**

- **MAE**: durchschnittlicher absoluter Fehler, gut interpretierbar und robuster gegenüber Ausreißern.
- **MSE/RMSE**: quadriert Fehler; große Fehler fallen stärker auf. RMSE hat wieder die Einheit der Zielgröße.
- **R²**: Anteil erklärter Varianz; höher ist besser.
- **Median Absolute Error**: sehr robust, da der Median genutzt wird.
- **MAPE**: relativer Fehler in Prozentnähe; problematisch bei Zielwerten nahe 0.

In [ ]:
# Synthetischer Mietpreis-Datensatz
rng = np.random.default_rng(RANDOM_STATE)
wohnflaeche = rng.normal(75, 18, 120).clip(25, 150)
zimmer = rng.integers(1, 6, 120)
entfernung = rng.uniform(0.5, 18, 120)
miete = 8.5 * wohnflaeche + 120 * zimmer - 18 * entfernung + rng.normal(0, 90, 120)
rent_df = pd.DataFrame({'Wohnfläche': wohnflaeche, 'Zimmer': zimmer, 'Entfernung Zentrum': entfernung, 'Miete': miete})
display(rent_df.head())

plt.scatter(rent_df['Wohnfläche'], rent_df['Miete'], alpha=0.75)
plt.xlabel('Wohnfläche')
plt.ylabel('Miete')
plt.title('Synthetische Mietpreise')
plt.show()

### Live-Demo: Rohdaten und Vorhersage gemeinsam verändern

Hier siehst du die Rohdaten und direkt daneben, wie sich eine einfache Vorhersagekurve verändert. Der Slider `baumtiefe` steuert die Komplexität eines Decision Trees: sehr kleine Werte underfitten, sehr große Werte passen sich stärker an die Trainingsdaten an.

In [ ]:
def mietpreis_modell_demo(rauschen=90, baumtiefe=3, testanteil=25):
    rng = np.random.default_rng(RANDOM_STATE)
    flaeche = rng.normal(75, 18, 140).clip(25, 150)
    # Nichtlinearer Anteil: große Wohnungen werden überproportional teurer.
    miete_demo = 280 + 9.0 * flaeche + 0.035 * flaeche**2 + rng.normal(0, rauschen, len(flaeche))
    X_demo = flaeche.reshape(-1, 1)
    Xtr, Xte, ytr, yte = train_test_split(X_demo, miete_demo, test_size=testanteil/100, random_state=RANDOM_STATE)
    model = DecisionTreeRegressor(max_depth=baumtiefe, random_state=RANDOM_STATE)
    model.fit(Xtr, ytr)
    grid = np.linspace(X_demo.min(), X_demo.max(), 250).reshape(-1, 1)
    pred_te = model.predict(Xte)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].scatter(Xtr[:, 0], ytr, alpha=0.7, label='Training')
    axes[0].scatter(Xte[:, 0], yte, alpha=0.7, label='Test')
    axes[0].plot(grid[:, 0], model.predict(grid), color='black', label='Vorhersage')
    axes[0].set_title('Rohdaten + Modellkurve')
    axes[0].set_xlabel('Wohnfläche'); axes[0].set_ylabel('Miete')
    axes[0].legend()
    axes[1].scatter(yte, pred_te, alpha=0.75)
    axes[1].plot([yte.min(), yte.max()], [yte.min(), yte.max()], 'r--')
    axes[1].set_title(f'Test: MAE={mean_absolute_error(yte, pred_te):.1f}, RMSE={np.sqrt(mean_squared_error(yte, pred_te)):.1f}')
    axes[1].set_xlabel('y true'); axes[1].set_ylabel('y pred')
    plt.tight_layout(); plt.show()

widgets.interact(
    mietpreis_modell_demo,
    rauschen=widgets.IntSlider(value=90, min=0, max=250, step=10),
    baumtiefe=widgets.IntSlider(value=3, min=1, max=12),
    testanteil=widgets.IntSlider(value=25, min=15, max=50, step=5),
);

In [ ]:
def regressionsmodell_demo(modell='RandomForestRegressor', baumtiefe=5, ridge_alpha=1.0):
    # Das Modell wird bei jeder Änderung neu trainiert. So sieht man echte Änderungen in der Vorhersage.
    if modell == 'LinearRegression':
        model = LinearRegression()
    elif modell == 'Ridge':
        model = Ridge(alpha=ridge_alpha, random_state=RANDOM_STATE)
    elif modell == 'DecisionTreeRegressor':
        model = DecisionTreeRegressor(max_depth=baumtiefe, random_state=RANDOM_STATE)
    else:
        model = RandomForestRegressor(n_estimators=80, max_depth=baumtiefe, random_state=RANDOM_STATE, n_jobs=-1)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    residuals = y_test - pred
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].scatter(y_test, pred, alpha=0.75)
    axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
    axes[0].set_xlabel('y true'); axes[0].set_ylabel('y pred')
    axes[0].set_title(f'{modell}: RMSE={np.sqrt(mean_squared_error(y_test, pred)):.1f}, R²={r2_score(y_test, pred):.2f}')
    axes[1].scatter(pred, residuals, alpha=0.75)
    axes[1].axhline(0, color='red', linestyle='--')
    axes[1].set_xlabel('Vorhersage'); axes[1].set_ylabel('Residuum')
    axes[1].set_title('Fehler nach Modelländerung')
    plt.tight_layout(); plt.show()
    return pred

# Eine Default-Vorhersage für spätere Zellen.
best_reg = models_reg['RandomForestRegressor']
y_pred = best_reg.predict(X_test)

widgets.interact(
    regressionsmodell_demo,
    modell=widgets.Dropdown(options=list(models_reg.keys()), value='RandomForestRegressor'),
    baumtiefe=widgets.IntSlider(value=5, min=1, max=12),
    ridge_alpha=widgets.FloatSlider(value=1.0, min=0.0, max=20.0, step=0.5),
);

### Live-Demo: Ausreißer beeinflussen Training und Vorhersage

Hier werden Ausreißer **in die Trainingsdaten** eingebaut und das Modell danach neu trainiert. So sieht man nicht nur größere Fehler, sondern auch, wie Ausreißer die gelernte Vorhersagefunktion verschieben können.

In [ ]:
def outlier_training_demo(anzahl_ausreisser=4, staerke=120, modell='LinearRegression'):
    rng = np.random.default_rng(RANDOM_STATE)
    Xtr = X_train.copy()
    ytr = y_train.copy()
    if hasattr(ytr, 'copy'):
        ytr = ytr.copy()
    outlier_idx = rng.choice(len(ytr), size=min(anzahl_ausreisser, len(ytr)), replace=False)
    ytr_array = ytr.to_numpy() if hasattr(ytr, 'to_numpy') else np.asarray(ytr).copy()
    ytr_array[outlier_idx] += staerke

    if modell == 'LinearRegression':
        model = LinearRegression()
    elif modell == 'Ridge':
        model = Ridge(alpha=2.0, random_state=RANDOM_STATE)
    else:
        model = DecisionTreeRegressor(max_depth=4, random_state=RANDOM_STATE)
    model.fit(Xtr, ytr_array)
    pred = model.predict(X_test)
    clean_model = LinearRegression().fit(X_train, y_train)
    clean_pred = clean_model.predict(X_test)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].scatter(y_test, pred, alpha=0.75, label='mit Outlier-Training')
    axes[0].scatter(y_test, clean_pred, alpha=0.35, label='sauberes Training')
    axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
    axes[0].set_xlabel('y true'); axes[0].set_ylabel('y pred')
    axes[0].set_title('Scatter: Vorhersage verschiebt sich')
    axes[0].legend()
    axes[1].scatter(pred, y_test - pred, alpha=0.75)
    axes[1].axhline(0, color='red', linestyle='--')
    axes[1].set_xlabel('Vorhersage'); axes[1].set_ylabel('Residuum')
    axes[1].set_title(f'MAE={mean_absolute_error(y_test, pred):.1f}, RMSE={np.sqrt(mean_squared_error(y_test, pred)):.1f}')
    plt.tight_layout(); plt.show()

widgets.interact(
    outlier_training_demo,
    anzahl_ausreisser=widgets.IntSlider(value=4, min=0, max=20),
    staerke=widgets.IntSlider(value=120, min=-250, max=350, step=25),
    modell=widgets.Dropdown(options=['LinearRegression', 'Ridge', 'DecisionTreeRegressor'], value='LinearRegression'),
);

### Aufgaben

1. Ändere `max_depth` beim Decision Tree und beobachte RMSE und R².
2. Welche Metrik würdest du für Mietpreise mit wenigen extrem teuren Wohnungen bevorzugen? Begründe kurz.

## 3. Überwachtes Lernen als Klassifikation

Klassifikation sagt Klassen voraus, zum Beispiel krank/gesund oder Ziffer 0 bis 9.

### Datensätze zuerst ansehen

- **Breast Cancer**: binäre Klassifikation. `malignant` und `benign` sind fachlich wichtig, deshalb sind Recall, Precision und Thresholds hier besonders anschaulich.
- **Digits**: kleine 8x8-Bilder der Ziffern 0 bis 9. Der Datensatz zeigt, dass Klassifikation auch für Bilder möglich ist, ohne direkt Deep Learning zu nutzen.

Vor den Metriken prüfen wir Beispielzeilen und Klassenverteilung. Ohne diesen Schritt kann eine hohe Accuracy leicht falsch interpretiert werden.

In [ ]:
cancer_preview = load_breast_cancer(as_frame=True)
display(cancer_preview.frame.head())
print('Klassen:', dict(enumerate(cancer_preview.target_names)))
print('Klassenverteilung:')
display(cancer_preview.frame['target'].value_counts().rename('Anzahl').to_frame())

plt.bar(cancer_preview.target_names, cancer_preview.frame['target'].value_counts().sort_index())
plt.title('Breast Cancer: Klassenverteilung')
plt.ylabel('Anzahl')
plt.show()

In [ ]:
cancer = load_breast_cancer(as_frame=True)
Xc, yc = cancer.data, cancer.target
Xc_train, Xc_test, yc_train, yc_test = train_test_split(Xc, yc, test_size=0.25, stratify=yc, random_state=RANDOM_STATE)

models_clf = {
    'LogisticRegression': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    'KNeighborsClassifier': make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    'DecisionTreeClassifier': DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE),
    'RandomForestClassifier': RandomForestClassifier(n_estimators=80, max_depth=5, random_state=RANDOM_STATE, n_jobs=-1),
}
rows=[]
for name, model in models_clf.items():
    model.fit(Xc_train, yc_train)
    pred = model.predict(Xc_test)
    proba = model.predict_proba(Xc_test)[:, 1]
    rows.append({'Modell': name, 'Accuracy': accuracy_score(yc_test, pred), 'Balanced Accuracy': balanced_accuracy_score(yc_test, pred),
                 'Precision': precision_score(yc_test, pred), 'Recall': recall_score(yc_test, pred), 'F1': f1_score(yc_test, pred),
                 'ROC AUC': roc_auc_score(yc_test, proba), 'PR AUC': average_precision_score(yc_test, proba), 'MCC': matthews_corrcoef(yc_test, pred)})
display(pd.DataFrame(rows).sort_values('F1', ascending=False))

logreg = models_clf['LogisticRegression']
yc_pred = logreg.predict(Xc_test)
ConfusionMatrixDisplay.from_predictions(yc_test, yc_pred, display_labels=cancer.target_names)
plt.title('Confusion Matrix')
plt.show()
print(classification_report(yc_test, yc_pred, target_names=cancer.target_names))

In [ ]:
# Digits kurz laden und visualisieren: Mehrklassen-Klassifikation.
digits = load_digits()
fig, axes = plt.subplots(1, 6, figsize=(8, 2))
for ax, image, label in zip(axes, digits.images[:6], digits.target[:6]):
    ax.imshow(image, cmap='gray')
    ax.set_title(f'Label {label}')
    ax.axis('off')
plt.suptitle('load_digits: kleine 8x8-Bilder')
plt.show()

In [ ]:
# Imbalanced Data Demo: Hier verändern wir Klassenverhältnis und Threshold.
def imbalanced_demo(positive_klasse_prozent=6, threshold=0.50):
    gewicht_pos = positive_klasse_prozent / 100
    Xi, yi = make_classification(
        n_samples=1200,
        n_features=8,
        n_informative=3,
        weights=[1 - gewicht_pos, gewicht_pos],
        flip_y=0.02,
        random_state=RANDOM_STATE,
    )
    Xi_train, Xi_test, yi_train, yi_test = train_test_split(Xi, yi, test_size=0.3, stratify=yi, random_state=RANDOM_STATE)
    majority_pred = np.zeros_like(yi_test)
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
    model.fit(Xi_train, yi_train)
    proba_i = model.predict_proba(Xi_test)[:, 1]
    pred_i = (proba_i >= threshold).astype(int)

    print('Nur Mehrheitsklasse vorhersagen')
    print('Accuracy:', round(accuracy_score(yi_test, majority_pred), 3))
    print('Recall positive Klasse:', round(recall_score(yi_test, majority_pred, zero_division=0), 3))
    print('Balanced Accuracy:', round(balanced_accuracy_score(yi_test, majority_pred), 3))
    print('Logistic Regression mit einstellbarem Threshold')
    print('Accuracy:', round(accuracy_score(yi_test, pred_i), 3), '| Recall:', round(recall_score(yi_test, pred_i), 3), '| Precision:', round(precision_score(yi_test, pred_i, zero_division=0), 3))

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].bar(['Klasse 0', 'Klasse 1'], np.bincount(yi_test))
    axes[0].set_title('Testdaten: Klassenverteilung')
    ConfusionMatrixDisplay.from_predictions(yi_test, pred_i, ax=axes[1], colorbar=False)
    axes[1].set_title(f'Confusion Matrix bei {threshold:.2f}')
    RocCurveDisplay.from_predictions(yi_test, proba_i, ax=axes[2])
    tn, fp, fn, tp = confusion_matrix(yi_test, pred_i).ravel()
    aktueller_fpr = fp / max(1, fp + tn)
    aktueller_tpr = tp / max(1, tp + fn)
    axes[2].scatter([aktueller_fpr], [aktueller_tpr], color='red', label='aktueller Threshold')
    axes[2].legend(loc='lower right')
    axes[2].set_title('ROC-Kurve')
    plt.tight_layout(); plt.show()

widgets.interact(
    imbalanced_demo,
    positive_klasse_prozent=widgets.IntSlider(value=6, min=2, max=40, step=2),
    threshold=widgets.FloatSlider(value=0.50, min=0.05, max=0.95, step=0.05),
);

### Live-Demo: Threshold, Recall und Fehlalarme

Bei medizinisch inspirierten Beispielen ist der Threshold besonders wichtig: Ein niedriger Threshold erhöht meist den Recall, erzeugt aber mehr False Positives. Der rote Punkt zeigt die aktuelle Position auf ROC- und Precision-Recall-Kurve.

In [ ]:
proba = logreg.predict_proba(Xc_test)[:, 1]
def threshold_demo(threshold=0.50):
    pred = (proba >= threshold).astype(int)
    fp = confusion_matrix(yc_test, pred)[0, 1]
    fn = confusion_matrix(yc_test, pred)[1, 0]
    print(f'Precision: {precision_score(yc_test, pred):.3f}')
    print(f'Recall:    {recall_score(yc_test, pred):.3f}')
    print(f'F1:        {f1_score(yc_test, pred):.3f}')
    print(f'False Positives: {fp} | False Negatives: {fn}')
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    ConfusionMatrixDisplay.from_predictions(yc_test, pred, display_labels=cancer.target_names, ax=axes[0], colorbar=False)
    axes[0].set_title(f'Matrix bei Threshold {threshold:.2f}')
    RocCurveDisplay.from_predictions(yc_test, proba, ax=axes[1])
    axes[1].scatter(
        [fp / max(1, ((yc_test == 0) & (pred == 0)).sum() + fp)],
        [recall_score(yc_test, pred)],
        color='red', label='aktueller Threshold'
    )
    axes[1].legend(loc='lower right')
    PrecisionRecallDisplay.from_predictions(yc_test, proba, ax=axes[2])
    axes[2].scatter([recall_score(yc_test, pred)], [precision_score(yc_test, pred)], color='red', label='aktueller Threshold')
    axes[2].legend(loc='lower left')
    plt.tight_layout(); plt.show()
widgets.interact(threshold_demo, threshold=widgets.FloatSlider(value=0.5, min=0.05, max=0.95, step=0.05));

**Wann welche Klassifikationsmetrik?**

- **Accuracy**: gut bei ungefähr gleich großen Klassen und ähnlichen Fehlerkosten.
- **Balanced Accuracy**: besser bei Klassenungleichgewicht.
- **Recall**: wichtig, wenn Verpassen teuer ist, z. B. medizinische Risiken.
- **Precision**: wichtig, wenn Fehlalarme teuer sind.
- **F1**: Kompromiss aus Precision und Recall.
- **ROC AUC**: allgemeine Trennfähigkeit über Thresholds.
- **PR AUC**: besonders hilfreich bei seltenen positiven Klassen.

### Aufgaben

1. Verschiebe den Threshold und beobachte False Positives und False Negatives.
2. Warum ist Accuracy bei der Imbalanced Demo irreführend?

## 4. Unüberwachtes Lernen

Beim Training werden keine Labels verwendet. Echte Labels nutzen wir nur nachträglich zur Analyse.

### Datensätze in diesem Kapitel

- **Wine**: chemische Messwerte verschiedener Weinsorten. Labels werden beim Clustering nicht verwendet, sondern nur danach zur Bewertung.
- **Digits**: Bilddaten können ebenfalls in 2D projiziert werden. Dadurch sieht man, dass PCA/Clustering nicht auf Tabellen mit klassischen Messwerten beschränkt ist.

In [ ]:
wine = load_wine(as_frame=True)
Xw = wine.data
yw = wine.target
Xw_scaled = StandardScaler().fit_transform(Xw)
pca = PCA(n_components=2, random_state=RANDOM_STATE)
Xw_pca = pca.fit_transform(Xw_scaled)

cluster_models = {
    'KMeans': KMeans(n_clusters=3, n_init=10, random_state=RANDOM_STATE),
    'Agglomerative': AgglomerativeClustering(n_clusters=3),
    'DBSCAN': DBSCAN(eps=2.2, min_samples=5),
}
rows=[]
for name, model in cluster_models.items():
    labels = model.fit_predict(Xw_scaled)
    mask = labels != -1
    sil = silhouette_score(Xw_scaled[mask], labels[mask]) if len(set(labels[mask])) > 1 else np.nan
    rows.append({'Modell': name, 'Silhouette': sil, 'ARI': adjusted_rand_score(yw, labels), 'NMI': normalized_mutual_info_score(yw, labels), 'Cluster': len(set(labels))})
display(pd.DataFrame(rows))

plt.scatter(Xw_pca[:, 0], Xw_pca[:, 1], c=yw, cmap='tab10')
plt.title('PCA-Projektion: echte Labels nur zur Ansicht')
plt.xlabel('PC1'); plt.ylabel('PC2'); plt.show()

### Live-Demo: KMeans

KMeans sucht Clusterzentren und ordnet jeden Punkt dem nächsten Zentrum zu. Mit dem Slider kannst du testen, was passiert, wenn die gewählte Clusteranzahl nicht zur Datenstruktur passt.


In [ ]:
def kmeans_widget(clusteranzahl=3):
    labels = KMeans(n_clusters=clusteranzahl, n_init=10, random_state=RANDOM_STATE).fit_predict(Xw_scaled)
    print('Silhouette:', round(silhouette_score(Xw_scaled, labels), 3), '| ARI:', round(adjusted_rand_score(yw, labels), 3), '| NMI:', round(normalized_mutual_info_score(yw, labels), 3))
    plt.scatter(Xw_pca[:, 0], Xw_pca[:, 1], c=labels, cmap='tab10')
    plt.title(f'KMeans mit {clusteranzahl} Clustern')
    plt.show()
widgets.interact(kmeans_widget, clusteranzahl=widgets.IntSlider(value=3, min=2, max=8));

### Live-Demo: DBSCAN

DBSCAN sucht dichte Regionen. `eps` steuert den Nachbarschaftsradius, `min_samples` die Mindestanzahl Punkte pro dichter Region. Zu kleine oder zu große Werte erzeugen schnell Rauschen oder einen einzigen großen Cluster.


In [ ]:
def dbscan_widget(eps=2.2, min_samples=5):
    labels = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(Xw_scaled)
    mask = labels != -1
    sil = silhouette_score(Xw_scaled[mask], labels[mask]) if len(set(labels[mask])) > 1 else np.nan
    print('Cluster inklusive Rauschen:', sorted(set(labels)), '| Silhouette:', sil)
    plt.scatter(Xw_pca[:, 0], Xw_pca[:, 1], c=labels, cmap='tab10')
    plt.title('DBSCAN: eps und min_samples steuern Dichte')
    plt.show()
widgets.interact(dbscan_widget, eps=widgets.FloatSlider(value=2.2, min=0.5, max=5.0, step=0.1), min_samples=widgets.IntSlider(value=5, min=2, max=15));

### Aufgaben

1. Stelle KMeans absichtlich auf 2 oder 6 Cluster. Was passiert mit ARI/NMI?
2. Suche DBSCAN-Parameter, bei denen fast alles als Rauschen (`-1`) markiert wird.

## 5. Bias-Varianz-Tradeoff

- **Bias**: Modell ist zu einfach und verfehlt die Grundstruktur.
- **Varianz**: Modell reagiert zu stark auf einzelne Trainingspunkte.
- **Underfitting**: hoher Trainings- und Validierungsfehler.
- **Overfitting**: niedriger Trainingsfehler, aber hoher Validierungsfehler.
- **Generalisierung**: gute Leistung auf neuen Daten.

In [ ]:
def true_complex_function(x):
    # Bewusst nicht nur Sinus: kubischer Trend plus lokale Wellen.
    return 0.12 * x**3 - 0.65 * x + 0.55 * np.cos(2.2 * x)

def make_nonlinear(n=80, noise=0.25):
    rng = np.random.default_rng(RANDOM_STATE)
    X = np.sort(rng.uniform(-3, 3, n)).reshape(-1, 1)
    y = true_complex_function(X[:, 0]) + rng.normal(0, noise, n)
    return X, y

def bias_variance_demo(polynomgrad=3, rauschen=0.25, trainingsdaten=80):
    Xn, yn = make_nonlinear(trainingsdaten, rauschen)
    Xtr, Xval, ytr, yval = train_test_split(Xn, yn, test_size=0.35, random_state=RANDOM_STATE)
    model = make_pipeline(PolynomialFeatures(polynomgrad), LinearRegression())
    model.fit(Xtr, ytr)
    grid = np.linspace(-3, 3, 300).reshape(-1, 1)
    true_y = true_complex_function(grid[:, 0])
    train_rmse = np.sqrt(mean_squared_error(ytr, model.predict(Xtr)))
    val_rmse = np.sqrt(mean_squared_error(yval, model.predict(Xval)))
    plt.scatter(Xtr, ytr, label='Training', alpha=0.8)
    plt.scatter(Xval, yval, label='Validierung', alpha=0.8)
    plt.plot(grid, true_y, 'g--', label='wahre Funktion')
    plt.plot(grid, model.predict(grid), color='black', label='Modell')
    ymin, ymax = np.percentile(np.r_[yn, true_y], [1, 99])
    pad = 0.5
    plt.ylim(ymin - pad, ymax + pad)
    plt.title(f'Grad {polynomgrad}: Train RMSE={train_rmse:.2f}, Val RMSE={val_rmse:.2f}')
    plt.legend(); plt.show()

widgets.interact(bias_variance_demo, polynomgrad=widgets.IntSlider(value=3, min=1, max=15), rauschen=widgets.FloatSlider(value=0.25, min=0.0, max=1.0, step=0.05), trainingsdaten=widgets.IntSlider(value=80, min=20, max=200, step=10));

### Underfitting, passende Komplexität und Overfitting sichtbar vergleichen

Die nächsten Plots zeigen dieselben Daten mit unterschiedlich komplexen Modellen: Grad 1 ist oft zu starr, mittlere Grade passen die Struktur sinnvoll an, sehr hohe Grade können einzelne Trainingspunkte auswendig lernen.


In [ ]:
Xn_base, yn_base = make_nonlinear(80, 0.25)
Xtr_base, Xval_base, ytr_base, yval_base = train_test_split(Xn_base, yn_base, test_size=0.35, random_state=RANDOM_STATE)
grid_base = np.linspace(-3, 3, 300).reshape(-1, 1)
true_base = true_complex_function(grid_base[:, 0])
ymin, ymax = np.percentile(np.r_[yn_base, true_base], [1, 99])
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, degree, title in zip(axes, [1, 4, 15], ['Underfitting', 'sinnvolle Komplexität', 'Overfitting']):
    model = make_pipeline(PolynomialFeatures(degree), LinearRegression())
    model.fit(Xtr_base, ytr_base)
    train_rmse = np.sqrt(mean_squared_error(ytr_base, model.predict(Xtr_base)))
    val_rmse = np.sqrt(mean_squared_error(yval_base, model.predict(Xval_base)))
    ax.scatter(Xtr_base, ytr_base, alpha=0.7, label='Training')
    ax.scatter(Xval_base, yval_base, alpha=0.7, label='Validierung')
    ax.plot(grid_base, true_base, 'g--', label='wahre Funktion')
    ax.plot(grid_base, model.predict(grid_base), color='black', label='Modell')
    ax.set_ylim(ymin - 0.6, ymax + 0.6)
    ax.set_title(f'{title}\nGrad {degree}: Train {train_rmse:.2f}, Val {val_rmse:.2f}')
    ax.set_xlabel('x')
axes[0].set_ylabel('y')
axes[0].legend()
plt.tight_layout(); plt.show()

### Overfitting wie in der Praxis: Training sehr gut, Test schlecht

In dieser Demo trainieren wir auf wenigen verrauschten Punkten und testen auf vielen neuen Punkten aus derselben Grundfunktion. Ein hoher Polynomgrad kann die Trainingspunkte fast auswendig lernen, generalisiert aber schlecht.

In [ ]:
def echtes_overfitting_demo(polynomgrad=14, trainingsdaten=25, rauschen=0.35):
    rng = np.random.default_rng(RANDOM_STATE)
    X_train_small = np.sort(rng.uniform(-3, 3, trainingsdaten)).reshape(-1, 1)
    y_train_small = true_complex_function(X_train_small[:, 0]) + rng.normal(0, rauschen, trainingsdaten)
    X_test_dense = np.linspace(-3, 3, 250).reshape(-1, 1)
    y_test_true = true_complex_function(X_test_dense[:, 0])
    model = make_pipeline(PolynomialFeatures(polynomgrad), LinearRegression())
    model.fit(X_train_small, y_train_small)
    pred_train = model.predict(X_train_small)
    pred_test = model.predict(X_test_dense)
    train_rmse = np.sqrt(mean_squared_error(y_train_small, pred_train))
    test_rmse = np.sqrt(mean_squared_error(y_test_true, pred_test))
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].scatter(X_train_small, y_train_small, label='verrauschte Trainingsdaten')
    axes[0].plot(X_test_dense, y_test_true, 'g--', label='wahre Funktion')
    axes[0].plot(X_test_dense, pred_test, color='black', label='Modell')
    ymin, ymax = np.percentile(np.r_[y_train_small, y_test_true], [1, 99])
    axes[0].set_ylim(ymin - 0.8, ymax + 0.8)
    axes[0].set_title(f'Grad {polynomgrad}: Train RMSE={train_rmse:.2f}, Test RMSE={test_rmse:.2f}')
    axes[0].legend()
    axes[1].scatter(y_test_true, pred_test, alpha=0.75)
    axes[1].plot([y_test_true.min(), y_test_true.max()], [y_test_true.min(), y_test_true.max()], 'r--')
    axes[1].set_xlim(y_test_true.min() - 0.5, y_test_true.max() + 0.5)
    axes[1].set_ylim(y_test_true.min() - 0.5, y_test_true.max() + 0.5)
    axes[1].set_xlabel('wahres y auf neuen Punkten')
    axes[1].set_ylabel('vorhergesagtes y')
    axes[1].set_title('Generalisation auf Testdaten')
    plt.tight_layout(); plt.show()

widgets.interact(
    echtes_overfitting_demo,
    polynomgrad=widgets.IntSlider(value=14, min=1, max=25),
    trainingsdaten=widgets.IntSlider(value=25, min=10, max=120, step=5),
    rauschen=widgets.FloatSlider(value=0.35, min=0.0, max=1.0, step=0.05),
);

In [ ]:
Xn, yn = make_nonlinear(160, 0.3)
model = make_pipeline(PolynomialFeatures(6), LinearRegression())
train_sizes, train_scores, val_scores = learning_curve(model, Xn, yn, cv=5, scoring='neg_root_mean_squared_error', train_sizes=np.linspace(0.2, 1.0, 5))
plt.plot(train_sizes, -train_scores.mean(axis=1), marker='o', label='Training')
plt.plot(train_sizes, -val_scores.mean(axis=1), marker='o', label='Validierung')
plt.xlabel('Anzahl Trainingsdaten')
plt.ylabel('RMSE')
plt.title('Learning Curve')
plt.legend(); plt.show()

### Aufgaben

1. Finde einen Polynomgrad, der sichtbar underfittet.
2. Erhöhe den Polynomgrad stark und reduziere die Trainingsdaten. Warum steigt die Varianz?

## 6. Metrikwahl in der Praxis

| Situation | Sinnvolle Metrik |
| --- | --- |
| Regression mit Ausreißern | MAE oder Median Absolute Error |
| Regression mit kritisch großen Fehlern | RMSE |
| Regression zur erklärten Varianz | R² |
| Klassifikation mit Klassenungleichgewicht | Balanced Accuracy, F1, PR AUC oder klassenspezifischer Recall |
| Medizinische Risikofälle | Recall für die positive Klasse |
| Hohe Fehlalarmkosten | Precision |

### Reflexionsaufgabe

Wähle ein eigenes Beispiel aus deinem Alltag. Welche Fehlerart wäre teurer: ein False Positive oder ein False Negative? Welche Metrik passt dazu?